# LLM Pipeline Profiler — Kaggle Notebook Example

This notebook demonstrates how to instrument an LLM inference pipeline using `llm-profiler`.

It is configured to run in **Mock GPU Mode** using the environment variable `LLM_PROFILER_MOCK_GPU=1`. This allows the notebook to run and output complete profiles (including GPU metrics) even on CPU-only Kaggle sessions or local development machines.

In [1]:
# 1. Install the profiler library
# In a real Kaggle notebook, you would run: !pip install llm-profiler
# For this local workspace check, we can install the package we just built.
!pip install llm-profiler

In [2]:
# 2. Setup the Profiler & Mock Environment
import os
import time

# Enable mock GPU so that it works seamlessly without NVML/GPU hardware
os.environ["LLM_PROFILER_MOCK_GPU"] = "1"

from llm_profiler import Tracer

# Initialize Tracer. 
# None = offline mode, which saves the results locally as 'run.json'
tracer = Tracer(run_name="kaggle-llama-demo-run", model_name="Llama-3-8B-Instruct", dashboard_url=None)

In [3]:
# 3. Stage 1: Model Loading
print("Loading model...")
with tracer.stage("model_load"):
    # Simulate loading a large model
    time.sleep(1.5)
print("Model loaded.")

Loading model...
Model loaded.


In [4]:
# 4. Stage 2: Tokenization
prompt = "Tell me a short story about an astronaut visiting antigravity fields."

@tracer.stage("tokenize")
def tokenize_prompt(text):
    print(f"Tokenizing prompt: {text}")
    time.sleep(0.3)  # Simulate tokenization process
    return [1, 25, 439, 2901, 48]

input_ids = tokenize_prompt(prompt)

Tokenizing prompt: Tell me a short story about an astronaut visiting antigravity fields.


In [5]:
# 5. Stage 3: Generation (Inference Loop)
print("Generating text...")
with tracer.stage("generate"):
    # Simulate autoregressive generation loop
    for i in range(12):
        time.sleep(0.1)
    
    # Log custom metric metadata (e.g., token throughput)
    tracer.log_metric("tokens_generated", 128.0)
    tracer.log_metric("batch_size", 1.0)
    tracer.log_metric("tps", 128.0 / 1.2)

print("Generation finished.")

Generating text...
Generation finished.


In [6]:
# 6. Stage 4: Postprocessing
print("Running post-processing...")
with tracer.stage("postprocess"):
    time.sleep(0.2)
print("Done.")

Running post-processing...
Done.


In [7]:
# 7. Display generated run.json profiling results
import json
with open("run.json", "r") as f:
    run_data = json.load(f)

print(json.dumps(run_data, indent=2)[:1500] + "\n... [TRUNCATED] ...")

{
  "id": "c2e227b0-0ef1-4ce9-961a-eeb06e79be47",
  "name": "kaggle-llama-demo-run",
  "created_at": "2026-07-16T06:47:42.509715",
  "model_name": "Llama-3-8B-Instruct",
  "hardware_info": {
    "cpu_count": 12,
    "gpu_type": "NVIDIA Tesla T4 (Mock)",
    "gpu_vram_mb": 15360.0
  },
  "stages": [
    {
      "id": "8161af8d-7281-4742-b4ae-57d9481eb3f7",
      "name": "model_load",
      "start_time": "2026-07-16T06:47:42.514255",
      "end_time": "2026-07-16T06:47:44.031119",
      "duration_ms": 1516.859099996509,
      "cpu_percent": 1.0333333333333334,
      "ram_used_mb": 199.3515625,
      "gpu_util_percent": 80.90961179472887,
      "gpu_mem_used_mb": 4713.698015823916,
      "metrics": [
        {
          "timestamp": "2026-07-16T06:47:42.516014",
          "key": "cpu_percent",
          "value": 0.0
        },
        {
          "timestamp": "2026-07-16T06:47:42.516014",
          "key": "ram_used_mb",
          "value": 199.30078125
        },
        {
          "times